<a href="https://colab.research.google.com/github/raisharad/GenerativeAIandAgenticAI/blob/main/Sharad_Rai__Module3_Starter_Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# <font color="blue"><b>From Instruction Fine-Tuning to QLoRA: A Cost–Quality Study on Financial Instruction Tuning</b></font>

---

## Problem Statement

Large language models are expensive to adapt. **Full fine-tuning** updates every weight in the model — it works well but needs a lot of GPU memory and compute, which puts it out of reach on modest hardware. Two parameter-efficient methods promise most of the quality for a fraction of the cost:

- **LoRA** freezes the model and trains only small low-rank adapter matrices (a small fraction of the parameters; the fraction depends on rank and target modules).
- **QLoRA** goes further — it stores the frozen model in **4-bit**, then trains the same LoRA adapters on top, cutting memory again.

In this assignment you will put these three methods on a real task: teaching **Qwen2.5-0.5B-Instruct** to answer financial and general instructions from the **Finance Alpaca (`gbharti/finance-alpaca`)** dataset (each example is `instruction (+ optional input) → output`). You will fine-tune the **same model on the same data three ways**, then measure what each approach costs and what it buys.

**The central question:** *given a fixed model, a fixed dataset, and a single free T4 GPU, is it worth updating all ~500M parameters — or does a low-rank slice (LoRA), or a low-rank slice over a 4-bit backbone (QLoRA), get you almost the same answer quality for far less memory?* You will also compare **two LoRA ranks (r = 8 and r = 16)** for both LoRA and QLoRA, to see how adapter capacity trades against cost.

---

## What You Will Do

You will build the pipeline end-to-end, as a hand-written training loop (a torch `Dataset`, a `collate` function, and `DataLoader`s — no `Trainer`), reused across all three methods:

1. **Imports & seed** — set up the environment reproducibly. *(provided)*
2. **Configuration** — model, data sizes, and all hyperparameters in one place. *(provided)*
3. **Helper functions** — LoRA/QLoRA configs, parameter counting, memory utilities. *(provided)*
4. **Load dataset + statistics** — take 300 / 30 / 30 for train / val / test; inspect fields and lengths.
5. **Dataset, collate & loaders** — build the prompt with the chat template, mark answer tokens with an offset-based mask, pad batches, and create the three loaders.
6. **Model load + IFT training** — write the masked-loss + shared training loop, then run **full instruction fine-tuning**.
7. **LoRA** — freeze the backbone and train adapters at **r = 8 and r = 16**.
8. **QLoRA** — repeat the rank sweep with a **4-bit NF4** backbone.
9. **Test evaluation** — greedy-generate on the test set and score every run with **ROUGE-L** and **BERTScore-F1**.
10. **Plots** — per-epoch train vs validation loss for every run.
11. **Error analysis** — inspect the best model's answers vs the references and summarise the failure modes.
12. **Parameter analysis** — compare quality against % trainable parameters, peak memory, and training time.

**How the notebook is organised**
- Cells marked **`# ── PROVIDED ──`** are already written — just run them in order.
- You only need to write the code inside **`## CODE HERE ##`**.
- Every step with a red **`[marks]`** tag is graded: complete the code where indicated and write the **`Your Answer`** reflection.
- Reflections must **cite the actual numbers your run produced** — not generic descriptions.

**Runtime target:** Colab **T4 GPU**. Use a small subset and short runs; session duration and memory depend on the installed software and runtime. Select a GPU and run top-to-bottom after a restart. No API key needed. Checkpoints are saved locally so only one run occupies GPU memory at a time.

> **Scope.** This is an educational instruction-tuning experiment. Similarity to a reference does not verify financial calculations, current regulations, or suitability of advice.

---

## Mark Distribution

| Step | Topic | Marks |
|------|-------|-------|
| 1 | Imports & seed | 0 *(provided)* |
| 2 | Configuration | 0 *(provided)* |
| 3 | Helper functions (LoRA / QLoRA) | 0 *(provided)* |
| 4 | Load dataset + statistics | 8 |
| 5 | Dataset, collate & loaders | 14 |
| 6 | Model load + IFT training | 20 |
| 7 | LoRA (r = 8 and r = 16) | 14 |
| 8 | QLoRA (r = 8 and r = 16) | 14 |
| 9 | Test evaluation (ROUGE-L + BERTScore) | 10 |
| 10 | Plots (train vs validation) | 6 |
| 11 | Error analysis | 8 |
| 12 | Parameter analysis | 6 |
| | **Total** | **100** |

Most marks sit where the core learning is: the training loop and full IFT (Step 6), and the data pipeline plus the two PEFT methods (Steps 5, 7, 8). Steps 1–3 are provided scaffolding and carry no marks.

---
**Dataset setting:** use the full mixed-domain Finance Alpaca dataset as requested. The dataset card describes a combination of Stanford Alpaca, FiQA, and approximately 1.3k custom GPT-3.5-generated pairs. It is not a finance-only benchmark; random-sample results reflect the mixed dataset. No topic filter is applied because the dataset does not provide reliable row-level topic/source labels.

**Dataset source:** [Finance Alpaca dataset card](https://huggingface.co/datasets/gbharti/finance-alpaca) and [requested viewer](https://huggingface.co/datasets/gbharti/finance-alpaca/viewer/default/train?row=0). Use `instruction`, `input`, and `output`; ignore the unused `text` column. References have mixed provenance and should not be treated as uniformly machine-generated or expert-verified.

**Length policy:** reserve 64 tokens of the training budget for answer content. Step 5 trims an overlong user message to fit the prompt budget, preserving the chat structure; the same prompt construction is used at evaluation. Long answers are right-truncated during training without inserting an artificial EOS. Inspect the reported truncation counts and discuss the information lost.

**Results:** run all five experiments before completing the reflection templates. No scores from an earlier dataset are retained.


In [1]:
# Install the training and evaluation dependencies in a fresh Colab runtime.
!pip install -q "transformers>=4.45,<5" "datasets>=2.20,<4" "peft>=0.13,<1" "accelerate>=0.34,<2" "bitsandbytes>=0.43,<1" evaluate rouge_score bert_score


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2025.3.0 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [2]:
# Avoid a torchao/T4 incompatibility that can surface with peft/quantised stacks.
!pip uninstall -y -q torchao

### <font color="blue"><b>Step 1: Imports and Seed Setting</b></font>  *(provided — just run)*

Import the libraries used throughout — PyTorch and `torch.utils.data`, Hugging Face `transformers` / `datasets` / `peft`, and plotting — and fix a single random seed across `random`, NumPy, and PyTorch (`seed_everything`) so every run is reproducible.

> `# ── PROVIDED ──` This cell is written for you. Run it and move on.

In [3]:
# ============================================================
# 1. Imports and Seed Setting
# ---PROVIDED ------------------------------------------------
# ============================================================
import os, gc, math, time, random, warnings, copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from datasets import load_dataset
from transformers import (AutoTokenizer, AutoModelForCausalLM,
                          BitsAndBytesConfig, get_cosine_schedule_with_warmup,
                          set_seed)
from peft import (LoraConfig, get_peft_model, prepare_model_for_kbit_training,
                  TaskType)

warnings.filterwarnings("ignore")

SEED = 42

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    set_seed(seed)

seed_everything()

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
from peft import PeftModel


Device: cuda
GPU: Tesla T4


### <font color="blue"><b>Step 2: Configuration</b></font>  *(provided — just run)*

All the knobs in one place: the model ID, dataset split sizes (`N_TRAIN` / `N_VAL` / `N_TEST`), the sequence-length budget (`MAX_LEN`), and the training hyperparameters — batch size, gradient accumulation, epochs, the two learning rates (`LR_FULL`, `LR_LORA`), and the LoRA ranks. The rest of the notebook reads from these constants, so change behaviour here rather than editing later cells.

> `# ── PROVIDED ──` This cell is written for you. Run it and move on.

In [4]:
# ============================================================
#  Step 2: Configuration
# ---PROVIDED ------------------------------------------------
# ============================================================
MODEL_ID   = "Qwen/Qwen2.5-0.5B-Instruct"

# ---- Data ----
DATASET_ID = "gbharti/finance-alpaca"
CHECKPOINT_DIR = "finance_alpaca_checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
N_TRAIN    = 300
N_VAL      = 30
N_TEST     = 30
MAX_LEN    = 256        # token budget per (prompt + answer) example
ANSWER_RESERVE = 64    # minimum answer-space budget when trimming long prompts

# ---- Training ----
BATCH_SIZE = 4          # small: fits a free T4 - if not fit please reduce it
GRAD_ACCUM = 1          # effective batch = BATCH_SIZE * GRAD_ACCUM
EPOCHS     = 3
WARMUP_RATIO  = 0.05
MAX_GRAD_NORM = 1.0

LR_FULL    = 1e-5       # full fine-tuning: small LR (all weights move)
LR_LORA    = 2e-4       # LoRA / QLoRA: frozen backbone -> larger LR is safe

# ---- LoRA / QLoRA ----
LORA_R       = 16
LORA_ALPHA   = 32
LORA_DROPOUT = 0.05

# ---- Evaluation ----
EVAL_N     = N_TEST         # number of test prompts to generate on when scoring

### <font color="blue"><b>Step 3: Helper Functions</b></font>  *(provided — just run)*

Small utilities reused across every method: `count_trainable` (trainable-vs-total parameter counts), `get_lora_config` / `get_bnb_config` (the shared LoRA and 4-bit NF4 quantisation settings), and `gpu_mem_gb` / `free_memory` (peak-memory reporting and VRAM cleanup between runs).

> `# ── PROVIDED ──` This cell is written for you. Run it and move on.

In [5]:
# ============================================================
#  Step 3: Helper Functions (LoRA / QLoRA)
# ---PROVIDED ------------------------------------------------
# ============================================================

def count_trainable(model):
    """Return (trainable, total, percent-trainable) parameter counts."""
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    return trainable, total, 100 * trainable / total


def get_lora_config():
    """Standard LoRA config for Qwen2.5 (attention + MLP projections)."""
    return LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"],
    )


def get_bnb_config():
    """4-bit NF4 quantisation config for QLoRA (fp16 compute for T4)."""
    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    )


def gpu_mem_gb():
    """Peak GPU memory used since the last reset, in GB."""
    return torch.cuda.max_memory_allocated() / 1e9 if torch.cuda.is_available() else 0.0


def free_memory(*objs):
    """Collect unused objects and clear the cache; delete caller references first."""
    for o in objs:
        try: del o
        except Exception: pass
    gc.collect()
    torch.cuda.empty_cache()
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

## <font color="blue"><b>Step 4: Load Finance Alpaca and Statistics</b></font> <font color="red">**[8 marks]**</font>

Load `gbharti/finance-alpaca`, split `train`. Keep `instruction` (task/question), `input` (optional context), and `output` (reference answer). Ignore the unused `text` column. Fill missing context with an empty string; do not erase actual financial input. Remove missing/blank instructions or answers, then deduplicate normalized **instruction + input pairs** before splitting.

Shuffle with the fixed seed and select **300 train / 30 validation / 30 test**. Print the fraction of non-empty inputs and token-length statistics (mean, median, p95, max), plus a histogram. This is a mixed-domain dataset, so report performance as Finance Alpaca instruction tuning rather than finance-only performance. Random splits may still share related paraphrases.

**Marks:** code **[4 marks]** — correct loading, cleaning, disjoint splits, statistics and histogram; reflection **[4 marks]** — see below.


In [6]:
 # ============================================================
#  Step 4: Load Dataset and Statistics
# ============================================================

raw = load_dataset(DATASET_ID, split="train")
df = raw.to_pandas()
required = {"instruction", "input", "output"}
assert required.issubset(df.columns), f"Unexpected schema: {list(df.columns)}"
df = df[["instruction", "input", "output"]].copy()
df = df.dropna(subset=["instruction", "output"])
df["input"] = df["input"].fillna("")
for col in ["instruction", "input", "output"]:
    df[col] = df[col].astype(str).str.strip()
df = df[(df["instruction"] != "") & (df["output"] != "")].copy()
# Deduplicate the actual user prompt, including any context.
normalized = lambda col: df[col].str.casefold().str.replace(r"\s+", " ", regex=True)
df["_prompt_key"] = list(zip(normalized("instruction"), normalized("input")))
df = df.drop_duplicates("_prompt_key")
assert len(df) >= N_TRAIN + N_VAL + N_TEST

# Shuffle once, then slice off the splits by index.
df = df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
## CODE HERE##

keys = [set(frame["_prompt_key"]) for frame in [train_df, val_df, test_df]]
assert not (keys[0] & keys[1] or keys[0] & keys[2] or keys[1] & keys[2])

# ---- Basic statistics on the training split ----
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

frac_with_input = (train_df["input"] != "").mean()

def combined_len(r):
    ## CODE HERE##

lens = train_df.apply(combined_len, axis=1).values
print(f"Rows with extra input : {frac_with_input:.1%}")
print(f"Token length  mean={lens.mean():.0f}  median={np.median(lens):.0f}  "
      f"p95={np.percentile(lens,95):.0f}  max={lens.max()}")

# Show one example so the format is concrete.
## CODE HERE##

plt.figure(figsize=(8, 4))
plt.hist(lens, bins=40, color="#2c6e9e", edgecolor="white")
plt.axvline(np.percentile(lens, 95), color="crimson", ls="--",
            label=f"p95 = {np.percentile(lens,95):.0f} tokens")
plt.title("Example lengths (instruction + input + output)")
plt.xlabel("tokens"); plt.ylabel("count"); plt.legend()
plt.tight_layout(); plt.show()

IndentationError: expected an indented block after function definition on line 36 (3638159288.py, line 39)

### ✍️ Step 4 Reflection <font color="red">[4 marks]</font>

**[Your Answer]** must cover:
- Explain `instruction`, optional `input`, and `output`; report the observed non-empty-input fraction. **[1 mark]**
- Report mean / median / p95 / max token lengths and justify `MAX_LEN` against the distribution and GPU budget. **[2 marks]**
- Explain the mixed domains and reference provenance, and why agreement with these answers does not establish financial correctness. **[1 mark]**

*Cite the actual numbers your Finance Alpaca run produced.*


<font color="green">ANSWER:

---------------------
---------------------
---------------------
</font>

## <font color="blue"><b>Step 5: Dataset, Collate, and DataLoaders</b></font>  <font color="red">**[14 marks]**</font>

Build the input pipeline as a hand-written `torch.utils.data.Dataset` (like the DPO demo), with tokenisation deferred to a `collate` function:

- **`build_prompt`** — renders the instruction plus optional context and trims overlong user content to reserve `ANSWER_RESERVE` tokens. Apply the same policy for training and generation.
- **`SFTDataset`** — holds `(prompt, answer)` text pairs. The prompt is rendered with the tokenizer's **chat template** (instruction → *user* turn, primed to answer).
- **`tokenize_example`** — tokenises the full `prompt + answer` **once** and uses the **offset mapping** to mark exactly where the answer begins, producing a `comp_mask` (1 = answer token, 0 = prompt token).
- **`collate`** — pads a batch to its longest sequence and returns `input_ids`, `attn`, and `comp_mask`.
- **Three DataLoaders** — `train_loader` (shuffled), `val_loader`, `test_loader`.

Finally, print the **first full training batch** to confirm the shapes and the answer/prompt masking.

**Marks:** code <font color="red">**[10 marks]**</font> — `SFTDataset` + chat-template prompt, offset-based `comp_mask`, correct padded `collate`, three loaders, and the batch print; reflection <font color="red">**[4 marks]**</font> — see below.

In [ ]:
# ============================================================
#  Step 5: Dataset, Collate, and DataLoaders
# ============================================================

SYSTEM_MSG = (
    "You are a helpful assistant for financial and general questions. "
    "Use the supplied context, explain calculations and assumptions clearly, "
    "and state when information is insufficient. Do not invent facts or guarantee returns."
)
PROMPT_STATS = {"calls": 0, "trimmed": 0}

def build_prompt(instruction, input_text):
    ## CODE HERE ##


class SFTDataset(Dataset):
    """Holds (prompt_text, answer_text) pairs; tokenisation is deferred to collate()."""
    ## CODE HERE##


def tokenize_example(prompt, answer):
    """
    Tokenise the full (prompt + answer) once and use the offset mapping to mark
    exactly where the answer begins. comp_mask = 1 on answer tokens, 0 on prompt.
    Tokenising once avoids token-merge drift at the boundary.
    """
    ##CODE HERE##



def collate(batch):
    """Pad a batch to its longest sequence. Returns input_ids, attn, comp_mask."""
    ##CODE HERE##

# ---- Three DataLoaders ----
##CODE HERE##

print("Prompt construction across train/val/test:", dict(PROMPT_STATS))
print(f"Train batches : {len(train_loader)}")
print(f"Val batches   : {len(val_loader)}")
print(f"Test batches  : {len(test_loader)}")

# ---- Print the first batch of the train loader ----
# ---- Print the first FULL batch of the train loader ----
input_ids, attn, comp_mask = next(iter(train_loader))
print("=== First training batch (full) ===")
print("input_ids shape :", tuple(input_ids.shape))
print("attn shape      :", tuple(attn.shape))
print("comp_mask shape :", tuple(comp_mask.shape))
print("Answer tokens (supervised):", int(comp_mask.sum()))

torch.set_printoptions(profile="full")   # don't truncate the tensors

print("\ninput_ids:\n", input_ids)
print("\nattn:\n", attn)
print("\ncomp_mask:\n", comp_mask)

torch.set_printoptions(profile="default")   # restore normal printing
# Inspect the actual chat-formatted training lengths and truncation rate.
formatted_lens = np.array([
    len(tokenizer(build_prompt(r["instruction"], r["input"]) + r["output"],
                  add_special_tokens=False)["input_ids"]) + 1
    for _, r in train_df.iterrows()
])
print("Formatted length mean / median / p95 / max:",
      formatted_lens.mean(), np.median(formatted_lens),
      np.percentile(formatted_lens, 95), formatted_lens.max())
print(f"Training examples exceeding MAX_LEN: {(formatted_lens > MAX_LEN).mean():.1%}")


### ✍️ Step 5 Reflection  <font color="red">**[4 marks]**</font>

**[Your Answer]** must cover:
- Why the prompt uses the tokenizer's **chat template** for an `-Instruct` model (special tokens must match what the model saw in pre-training). <font color="red">**[1]**</font>
- What `comp_mask` encodes and why we compute the loss on the **answer tokens only** (completion-only). <font color="red">**[1]**</font>
- Why tokenising `prompt + answer` **once with offsets** is more correct than tokenising them separately and concatenating (avoids token-merge drift at the boundary). <font color="red">**[1]**</font>
- From the printed batch: the supervised-vs-masked token counts, and why padding fills `input_ids` with the pad token but `comp_mask` with 0. <font color="red">**[1]**</font>

*Cite the actual numbers your run produced.*

<font color="green">ANSWER:

---------------------
---------------------
---------------------
</font>

## <font color="blue"><b>Step 6: Model Loading and Full Instruction Fine-Tuning (IFT)</b></font>  <font color="red">**[20 marks]**</font>

This is the core of the notebook. First define the training machinery that all three methods reuse, then run full fine-tuning:

- **`batch_loss`** — the causal-LM next-token loss, but computed over **answer tokens only** (using `comp_mask`), so the model is never scored on re-typing the prompt.
- **`eval_loss`** — averages `batch_loss` over a whole loader (no gradients), used for validation.
- **`train_model`** — the shared manual training loop: gradient accumulation, warmup → cosine schedule, gradient clipping, per-epoch train/val history, and early stopping on validation loss. Returns the best model plus its history, training time, and peak memory.

Then load Qwen2.5-0.5B in **fp32** (pure fp16 full fine-tuning overflows on this model), confirm ~100% of parameters are trainable, run **full IFT** with the small learning rate `LR_FULL`, and print the per-epoch train/val losses.

**Marks:** code <font color="red">**[14 marks]**</font> — correct answer-only `batch_loss`, the `train_model` loop (accumulation, schedule, clipping, early stopping, history), fp32 model load, and the IFT run with per-epoch report; reflection <font color="red">**[6 marks]**</font> — see below.

In [ ]:
# ============================================================
#  Step 6: Load Model and Full Instruction Fine-Tuning (IFT)
# ============================================================

# ---- Loss over ANSWER tokens only (comp_mask == 1) ----
def batch_loss(model, input_ids, attn, comp_mask):
    """Mean next-token NLL over answer tokens for ONE batch."""
    ## CODE HERE ##

@torch.no_grad()
def eval_loss(model, loader):
    """Token-weighted mean NLL over ALL answer tokens in the loader (no gradients)."""
    ##CODE HERE##


# ---- Shared training loop (reused for IFT, LoRA, QLoRA) ----
def train_model(model, lr, tag):
    """
    Manual training loop: gradient accumulation, warmup->cosine schedule,
    gradient clipping, and early stopping on validation loss.
    Returns (best_model, hist, train_time_s, peak_mem_gb).
    hist keys: 'train_loss' & 'val_loss' (per epoch), 'step_loss' (per step).
    """
    seed_everything()
    free_memory()

    ## CODE HERE ##

    if best_state is not None:
        # strict=False: we restore only the trainable subset (adapters / full weights).
        model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()}, strict=False)

    train_time, peak_mem = time.time() - t0, gpu_mem_gb()
    print(f"[{tag}] best epoch = {best_epoch} | time = {train_time:.1f}s | peak mem = {peak_mem:.2f} GB")
    return model, hist, train_time, peak_mem


# ---- Load model for Full IFT ----
# fp32 (not fp16): pure-fp16 full fine-tuning overflows on Qwen2.5 and gives a
# garbage loss in the hundreds. fp32 is fine for a 0.5B model on a T4.
seed_everything()
free_memory()

full_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float32).to(DEVICE)
full_model.config.use_cache = False               # required with gradient checkpointing
full_model.gradient_checkpointing_enable()

tr, tot, pct_full = count_trainable(full_model)
print(f"[Full IFT] trainable = {tr:,} / {tot:,}  ({pct_full:.2f}%)\n")

# ---- Train ----
full_model, full_hist, full_time, full_mem = train_model(full_model, LR_FULL, "Full IFT")

# ---- Per-epoch loss history ----
print("\nEpoch |  Train Loss | Val Loss")
print("-" * 34)
for ep, (tl, vl) in enumerate(zip(full_hist["train_loss"], full_hist["val_loss"]), 1):
    print(f"  {ep:2d}  |   {tl:.4f}   | {vl:.4f}")
full_path = os.path.join(CHECKPOINT_DIR, "full_ift")
full_model.save_pretrained(full_path)
del full_model
free_memory()


### ✍️ Step 6 Reflection  <font color="red">**[6 marks]**</font>

**[Your Answer]** must cover:
- How `batch_loss` supervises **only** the answer tokens (the `comp_mask` shift + masked mean), and how completion-only loss differs from training on the whole sequence. <font color="red">**[2 marks]**</font>
- Where warmup, cosine decay, gradient clipping, and gradient accumulation appear in `train_model`, and your **effective batch size** (`BATCH_SIZE × GRAD_ACCUM`). <font color="red">**[2 marks]**</font>
- Why full IFT is loaded in **fp32** (pure fp16 full fine-tuning overflows on Qwen2.5) and why its learning rate `LR_FULL = 1e-5` is deliberately small. <font color="red">**[1 marks]**</font>
- The per-epoch train/val trend — is the model improving or starting to overfit? <font color="red">**[1 marks]**</font>

*Cite the actual numbers your run produced.*

<font color="green">ANSWER:

---------------------
---------------------
---------------------
</font>

## <font color="blue"><b>Step 7: LoRA Fine-Tuning — r = 8 and r = 16</b></font>  <font color="red">**[14 marks]**</font>

Now freeze the backbone and train only small low-rank adapter matrices. For a frozen weight `W`, LoRA learns `A` and `B` and adds `(α/r)·B·A` as a correction — so only a tiny fraction of the parameters are trained.

Run LoRA at **two ranks, r = 8 and r = 16**, holding the `alpha/rank` ratio fixed (`alpha = 2·r`) so you isolate the effect of *rank* alone. Reuse the same `train_model` loop, store each run in `lora_runs[rank]`, and report the trainable-parameter percentage and per-epoch losses for both.

**Marks:** code <font color="red">**[8 marks]**</font> — correct `LoraConfig` (rank, alpha, dropout, target modules), the two-rank sweep reusing `train_model`, and the trainable-% + per-epoch report for each rank; reflection <font color="red">**[6 marks]**</font> — see below.

In [ ]:
# ============================================================
#  Step 7: LoRA Fine-Tuning (rank sweep: r = 8 and r = 16)
# ============================================================
LORA_RANKS   = [8, 16]   # compare two LoRA ranks; alpha = 2 * rank keeps alpha/r = 2
LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj",
                "gate_proj", "up_proj", "down_proj"]

def build_lora_model(rank):
    """Fresh fp32 backbone (frozen) + LoRA adapters of the given rank."""
    seed_everything()
    free_memory()
    ##CODE HERE##
    return get_peft_model(base, cfg)


# ---- Run LoRA at each rank ----
lora_runs = {}   # rank -> dict(path, hist, time, mem, pct)

for rank in LORA_RANKS:
    ##CODE HERE##
    print("\nEpoch |  Train Loss | Val Loss")
    print("-" * 34)
    for ep in range(len(hist["train_loss"])):
        print(f"  {ep + 1:2d}  |   {hist['train_loss'][ep]:.4f}   | {hist['val_loss'][ep]:.4f}")
    print()

    del model
    free_memory()                            # clear the GPU before the next rank

### ✍️ Step 7 Reflection  <font color="red">**[6 marks]**</font>

**[Your Answer]** must cover:
- The role of the **rank `r`** and **alpha `α`** (the `α/r` scaling), and why `B` is zero-initialised (the adapter update starts at zero, so training is stable from step 0). <font color="red">**[2 marks]**</font>
- The **% trainable parameters** at r = 8 vs r = 16 (from `print_trainable_parameters`), and how both compare to full IFT's ~100%. <font color="red">**[2 marks]**</font>
- The effect of doubling the rank on validation loss — did r = 16 actually help, or just cost more parameters for the same quality? Cite both runs. <font color="red">**[1 marks]**</font>
- Why LoRA can safely use a much larger learning rate (`LR_LORA = 2e-4`) than full IFT. <font color="red">**[1 marks]**</font>

*Cite the actual numbers your run produced.*

<font color="green">ANSWER:

---------------------
---------------------
---------------------
</font>

## <font color="blue"><b>Step 8: QLoRA Fine-Tuning — r = 8 and r = 16</b></font>  <font color="red">**[14 marks]**</font>

LoRA froze the backbone but still kept it in full precision, which dominates GPU memory. **QLoRA** stores that frozen backbone in **4-bit NF4** instead, then trains the *same* LoRA adapters on top — the only thing that changes from Step 7 is the backbone's precision, making this a clean controlled comparison.

Set the 4-bit recipe with `BitsAndBytesConfig` (NF4, double quantisation, fp16 compute for the T4), call `prepare_model_for_kbit_training` before attaching the adapters, and run the **same two-rank sweep (r = 8 and r = 16)**. Store each run in `qlora_runs[rank]` and report the trainable-parameter percentage and per-epoch losses.

**Marks:** code <font color="red">**[8 marks]**</font> — correct `BitsAndBytesConfig`, `prepare_model_for_kbit_training`, the same `LoraConfig` as Step 7, the two-rank sweep reusing `train_model`, and the trainable-% + per-epoch report; reflection <font color="red">**[6 marks]**</font> — see below.

In [ ]:
# ============================================================
#  Step 8: QLoRA Fine-Tuning (4-bit NF4 backbone; rank sweep: r = 8 and r = 16)
# ============================================================

def build_qlora_model(rank):
    """Fresh 4-bit NF4 backbone (frozen) + the same LoRA adapters as Step 7."""
    seed_everything()
    free_memory()

   ##CODE HERE##
    return get_peft_model(base, cfg)


# ---- Run QLoRA at each rank ----
qlora_runs = {}   # rank -> dict(path, hist, time, mem, pct)

for rank in LORA_RANKS:
    ##CODE HERE##
    print("\nEpoch |  Train Loss | Val Loss")
    print("-" * 34)
    for ep in range(len(hist["train_loss"])):
        print(f"  {ep + 1:2d}  |   {hist['train_loss'][ep]:.4f}   | {hist['val_loss'][ep]:.4f}")
    print()

    del model
    free_memory()                               # clear the GPU before the next rank

### ✍️ Step 8 Reflection  <font color="red">**[6 marks]**</font>

**[Your Answer]** must cover:
- Each `BitsAndBytesConfig` setting in plain language — `load_in_4bit`, `nf4`, double quantisation, and `compute_dtype=float16` — and why compute is fp16 on a T4. <font color="red">**[2 marks]**</font>
- What `prepare_model_for_kbit_training` does and why it is required before attaching LoRA. <font color="red">**[1 marks]**</font>
- QLoRA vs LoRA at the **same rank**: how much peak memory did 4-bit save, and did it cost any measurable answer quality? Cite both. <font color="red">**[2 marks]**</font>
- Why QLoRA's trainable-parameter count matches LoRA's at the same rank (only the frozen backbone's precision changed). <font color="red">**[1 marks]**</font>

*Cite the actual numbers your run produced.*

<font color="green">ANSWER:

---------------------
---------------------
---------------------
</font>

## <font color="blue"><b>Step 9: Test Evaluation — IFT, LoRA, QLoRA</b></font>  <font color="red">**[10 marks]**</font>

Now measure answer quality on the held-out test set. Greedy-generate an answer for a fixed slice of test prompts (`EVAL_N`, the same prompts for every model so scores are comparable), then score the generations against the references with two complementary metrics:

- **ROUGE-L** — overlap of word sequences (did we *say* similar things?).
- **BERTScore-F1** — embedding similarity (did we *mean* similar things, even with different wording?).

Score every run — full IFT, both LoRA ranks, and both QLoRA ranks — and collect the numbers (plus each run's % trainable, peak memory, and training time) into `results`, with the generated text kept in `preds_store` for the error analysis.

**Marks:** code <font color="red">**[6 marks]**</font> — correct `generate_answers` (greedy, prompt-only decode), ROUGE-L + BERTScore scoring, and scoring all five runs into `results` / `preds_store`; reflection <font color="red">**[4 marks]**</font> — see below.

In [ ]:
# ============================================================
#  Step 9: Test Evaluation — IFT, LoRA, QLoRA
# ============================================================
import evaluate as hf_evaluate
rouge = hf_evaluate.load("rouge")

@torch.no_grad()
def generate_answers(model, frame, max_new_tokens=200, n=EVAL_N):
    """Greedy-decode an answer for each prompt. Returns (predictions, references)."""
    ## CODE HERE##
    return preds, refs


def score(preds, refs):
    """ROUGE-L (word overlap) + BERTScore-F1 (meaning similarity)."""
    ##CODE HERE##


results, preds_store = {}, {}   # name -> metrics ; name -> predictions

In [ ]:
# ---- Full IFT: reload the selected checkpoint ----
full_model = AutoModelForCausalLM.from_pretrained(full_path, torch_dtype=torch.float32).to(DEVICE)
full_preds, test_refs = generate_answers(full_model, test_df)
results["full_ift"] = {**score(full_preds, test_refs),
                       "pct_trainable": round(pct_full, 2),
                       "peak_mem_gb": round(full_mem, 2),
                       "train_time_s": round(full_time, 1)}
preds_store["full_ift"] = full_preds
print("[full_ift]", results["full_ift"])
del full_model
free_memory()


In [ ]:
# ---- LoRA and QLoRA, each at both ranks ----
for method, runs in [("lora", lora_runs), ("qlora", qlora_runs)]:
    for rank, run in runs.items():
        name = f"{method}_r{rank}"
        if method == "qlora":
            base = AutoModelForCausalLM.from_pretrained(
                MODEL_ID, quantization_config=get_bnb_config(), device_map={"": 0})
            # Match training: non-quantised layers back to fp32, as in Step 8
            base = prepare_model_for_kbit_training(base, use_gradient_checkpointing=False)
        else:
            base = AutoModelForCausalLM.from_pretrained(
                MODEL_ID, torch_dtype=torch.float32).to(DEVICE)
        eval_model = PeftModel.from_pretrained(base, run["path"])
        p, _ = generate_answers(eval_model, test_df)
        results[name] = {**score(p, test_refs),
                         "pct_trainable": round(run["pct"], 2),
                         "peak_mem_gb": round(run["mem"], 2),
                         "train_time_s": round(run["time"], 1)}
        preds_store[name] = p
        print(f"[{name}]", results[name])
        del eval_model, base
        free_memory()

### ✍️ Step 9 Reflection  <font color="red">**[4 marks]**</font>

**[Your Answer]** must cover:
- What ROUGE-L and BERTScore-F1 each measure, and why reporting both is more informative than either alone for free-form financial and general answers. <font color="red">**[2 marks]**</font>
- The scores across the five runs — which scores highest, and by how much over the others? <font color="red">**[1 marks]**</font>
- Given the mixed reference provenance, why you should weigh *relative* differences between methods more than the *absolute* score values. <font color="red">**[1 marks]**</font>

*Cite the actual numbers your run produced.*

<font color="green">ANSWER:

---------------------
---------------------
---------------------
</font>

## <font color="blue"><b>Step 10: Plots — Train vs Validation Loss</b></font>  <font color="red">**[6 marks]**</font>

Visualise how each run trained. Draw one panel per run — full IFT, both LoRA ranks, and both QLoRA ranks — showing per-epoch **train** and **validation** loss, all sharing the same y-axis so loss levels are directly comparable across methods and ranks. The curves come from each run's `hist` (`train_loss` / `val_loss`).

**Marks:** code <font color="red">**[4 marks]**</font> — one train-vs-val panel per run, shared y-axis, correct labels/legend/titles; reflection <font color="red">**[2 marks]**</font> — see below.

In [ ]:
# ============================================================
#  Step 10: Plots — Train vs Validation Loss (per epoch)
# ============================================================

# Gather every run's history under one name -> hist mapping.
plot_hist = {"full_ift": full_hist}
for method, runs in [("lora", lora_runs), ("qlora", qlora_runs)]:
    for rank, run in runs.items():
        plot_hist[f"{method}_r{rank}"] = run["hist"]

n = len(plot_hist)
fig, axes = plt.subplots(1, n, figsize=(4.2 * n, 4.2), sharey=True)
if n == 1:
    axes = [axes]

for ax, (name, hist) in zip(axes, plot_hist.items()):
    epochs = range(1, len(hist["train_loss"]) + 1)
    ax.plot(epochs, hist["train_loss"], "o-", label="Train")
    ax.plot(epochs, hist["val_loss"],   "s-", label="Validation")
    ax.set_title(name)
    ax.set_xlabel("Epoch")
    ax.set_xticks(list(epochs))
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)

axes[0].set_ylabel("Masked LM loss")
plt.suptitle("Training vs Validation Loss (per epoch)")
plt.tight_layout()
plt.show()

### ✍️ Step 10 Reflection  <font color="red">**[2 marks]**</font>

**[Your Answer]** must cover:
- Which method converged fastest and to the lowest validation loss, and whether any run shows signs of **overfitting** (validation rising while training keeps falling). <font color="red">**[1 marks]**</font>
- Whether a lower *training* loss reliably predicted better *test* answers back in Step 9 — and what that tells you. <font color="red">**[1 marks]**</font>

*Cite the curves and the Step 9 numbers.*

<font color="green">ANSWER:

---------------------
---------------------
---------------------
</font>

## <font color="blue"><b>Step 11: Error Analysis</b></font>  <font color="red">**[8 marks]**</font>

Numbers alone don't tell you *how* a model is wrong. Take the best run by **ROUGE-L**, print a few of its generated answers next to the references, and back that up with a quantitative summary of the failure modes: predicted-vs-reference length, the length ratio (a quick proxy for rambling or truncation), and counts of empty / much-shorter / much-longer generations.

**Marks:** code <font color="red">**[4 marks]**</font> — select the best run by ROUGE-L, show sample generations vs references, and compute the length-ratio + failure-mode summary; reflection <font color="red">**[4 marks]**</font> — see below.

In [ ]:
# ============================================================
#  Step 11: Error Analysis
# ============================================================

# Pick the best run by ROUGE-L.
best = max(results, key=lambda k: results[k]["rougeL"])
print(f"Best model by ROUGE-L: {best}  "
      f"(ROUGE-L={results[best]['rougeL']}, BERTScore-F1={results[best]['bertscore_f1']})")
print("=" * 76)

# Show a few generated answers vs the references.
for i in range(3):
    ##CODE HERE##

In [ ]:
# ---- Quantitative summary of the errors ----
##CODE HERE##

print(f"\nError summary for '{best}'  (over {len(best_preds)} test prompts)")
print("-" * 60)
print(f"Avg predicted length : {pred_lens.mean():.0f} tokens")
print(f"Avg reference length : {ref_lens.mean():.0f} tokens")
print(f"Length ratio (pred/ref): mean={len_ratio.mean():.2f}  median={np.median(len_ratio):.2f}")
print(f"Empty generations      : {empty}")
print(f"Much shorter (<0.5x)   : {too_short}")
print(f"Much longer  (>2.0x)   : {too_long}")

### ✍️ Step 11 Reflection  <font color="red">**[4 marks]**</font>

**[Your Answer]** must cover:
- One recurring way the model's answers differ from the references — length, specificity, hedging, or incorrect arithmetic, invented financial facts, missing assumptions, or unsupported certainty — with a concrete example from the samples. <font color="red">**[2 marks]**</font>
- What the length-ratio and failure counts reveal about the model's behaviour (rambling, truncation, empty generations). <font color="red">**[1 marks]**</font>
- One concrete change you would try to reduce these errors (e.g. more epochs, higher rank, longer `MAX_LEN`, or different decoding settings). <font color="red">**[1 marks]**</font>

*Cite the samples and the summary numbers your run produced.*

<font color="green">ANSWER:

---------------------
---------------------
---------------------
</font>

## <font color="blue"><b>Step 12: Parameter Analysis</b></font>  <font color="red">**[6 marks]**</font>

Bring everything together into the cost–quality picture. Build one comparison table across all five runs — **quality** (ROUGE-L, BERTScore-F1) against **cost** (% trainable parameters, peak GPU memory, training time) — then draw the supporting charts: quality bars with peak memory annotated, and a trainable-% bar chart that makes the full-IFT-vs-adapter gap obvious.

**Marks:** code <font color="red">**[4 marks]**</font> — the comparison table over all runs and the quality + trainable-% charts; reflection <font color="red">**[2 marks]**</font> — see below.

In [ ]:
# ============================================================
#  Step 12: Parameter Analysis
# ============================================================

# ---- Comparison table: quality vs cost across every run ----
##CODE HERE##

In [ ]:
# ---- Quality chart with peak memory annotated ----
names   = list(results.keys())
rougeLs = [results[m]["rougeL"] for m in names]
berts   = [results[m]["bertscore_f1"] for m in names]
x = np.arange(len(names)); w = 0.38

fig, ax = plt.subplots(figsize=(1.8 * len(names) + 3, 6))
ax.bar(x - w/2, rougeLs, w, label="ROUGE-L", color="#1a3c5e")
ax.bar(x + w/2, berts,   w, label="BERTScore-F1", color="#2c6e9e")
ax.set_xticks(x); ax.set_xticklabels(names, rotation=15)
ax.set_ylabel("score")
ax.set_title("Answer quality by method (peak GPU memory annotated)")

for i, m in enumerate(names):
    mem = results[m]["peak_mem_gb"]
    ax.text(i, max(rougeLs[i], berts[i]) + 0.01,
            f"{mem:.1f} GB" if mem is not None else "-",
            ha="center", fontsize=9, color="#444")

ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
# ---- Trainable-parameters comparison (the LoRA/QLoRA story) ----
fig, ax = plt.subplots(figsize=(1.6 * len(names) + 3, 5))
pcts = [results[m]["pct_trainable"] for m in names]
bars = ax.bar(names, pcts, color="#8452dd")
ax.set_ylabel("% trainable parameters")
ax.set_title("Trainable parameters by method")
ax.set_xticklabels(names, rotation=15)
for b, p in zip(bars, pcts):
    ax.text(b.get_x() + b.get_width()/2, p, f"{p:.2f}%", ha="center", va="bottom", fontsize=9)
plt.tight_layout(); plt.show()

### ✍️ Step 12 Reflection  <font color="red">**[2 marks]**</font>

**[Your Answer]** must cover:
- Which method gives the best **balance** of quality, trainable-%, memory, and training time for a free T4 — not simply the highest score — citing the comparison table. <font color="red">**[1 marks]**</font>
- How **rank** (8 vs 16) and **quantisation** (LoRA vs QLoRA) each traded answer quality against cost. <font color="red">**[1 marks]**</font>

*Cite the actual numbers from your comparison table.*

<font color="green">ANSWER:

---------------------
---------------------
---------------------
</font>

In [ ]:
# ----------------------END of the SOLUTION-----------------